# N03 · From Stellar Emission to Galaxy Spectra
## Blackbody radiation, stellar spectral libraries, population synthesis, and observed galaxy spectra

---

## Learning objectives

After this tutorial you will be able to:

1. Derive and plot the Planck blackbody function $B_\lambda(T)$ and $B_\nu(T)$, and relate them through Wien's law and the Stefan–Boltzmann law.
2. Use Wien's displacement law to infer a star's temperature from its peak emission wavelength, and vice versa.
3. Use the Stefan–Boltzmann law to compute the total luminosity of a star from its radius and effective temperature.
4. Compare Planck blackbody spectra to real observed stellar spectra from the MILES library, and identify characteristic absorption features (Balmer series, Ca H&K, G-band, Mg b, Na D, TiO bands).
5. Understand the Initial Mass Function (IMF) and the concept of a Simple Stellar Population (SSP); compute the integrated SSP spectrum by summing over an IMF using both blackbody and empirical MILES stellar templates.
6. Load stacked SDSS DR16 galaxy spectra (star-forming ELG and quiescent LRG) at rest-frame $z=0$ and identify their characteristic spectral features (emission lines, 4000 Å break, absorption lines).
7. Fit single-age blackbody and MILES SSP models to observed galaxy templates by $\chi^2$ minimisation over SSP age and interpret the best-fit ages physically.
8. Perform a penalized pixel-fitting (`ppxf`) analysis of the LRG spectrum using MILES stellar templates to extract the stellar line-of-sight velocity $V$ and velocity dispersion $\sigma$.

**Estimated time:** 5–7 hours

---

## 1. Theoretical background

### 1.1 The Planck blackbody function

A **blackbody** (or black body) is an idealised object that absorbs all incident radiation and emits radiation with a spectrum determined solely by its temperature $T$. Stars are well approximated as blackbodies to first order.

See: [Wikipedia — Black-body radiation](https://en.wikipedia.org/wiki/Black-body_radiation)

Max Planck (1900) derived the spectral radiance (energy emitted per unit time, per unit area, per unit solid angle, per unit frequency interval):

$$B_\nu(T) = \frac{2h\nu^3}{c^2} \frac{1}{e^{h\nu/(k_B T)} - 1}$$

where:
- $h = 6.626\,070\,15 \times 10^{-34}$ J s — Planck's constant *(exact, 2019 SI definition)*
- $c = 2.997\,924\,58 \times 10^8$ m s$^{-1}$ — speed of light in vacuum *(exact)*
- $k_B = 1.380\,649 \times 10^{-23}$ J K$^{-1}$ — Boltzmann constant *(exact, 2019 SI definition)*
- $\nu$ — photon frequency [Hz]
- $T$ — absolute temperature [K]

Units of $B_\nu$: W m$^{-2}$ Hz$^{-1}$ sr$^{-1}$.

The equivalent expression in **wavelength** is obtained via $|B_\lambda\,d\lambda| = |B_\nu\,d\nu|$ and $\nu = c/\lambda$:

$$\boxed{B_\lambda(T) = \frac{2hc^2}{\lambda^5} \frac{1}{e^{hc/(\lambda k_B T)} - 1}}$$

Units of $B_\lambda$: W m$^{-2}$ m$^{-1}$ sr$^{-1}$ = W m$^{-3}$ sr$^{-1}$.

**Note:** $B_\nu$ and $B_\lambda$ peak at *different* wavelengths because of the Jacobian of the coordinate transformation.

### 1.2 Wien's displacement law

See: [Wikipedia — Wien's displacement law](https://en.wikipedia.org/wiki/Wien%27s_displacement_law)

Differentiating $B_\lambda$ with respect to $\lambda$ and setting the result to zero yields:

$$\lambda_\mathrm{max} \cdot T = b_W \equiv 2.897\,771\,955 \times 10^{-3} \text{ m·K}$$

This is **Wien's displacement law** (NIST CODATA 2022; $b_W$ is derived from the exact constants $h$, $c$, $k_B$). The peak shifts to shorter wavelengths for hotter stars:
- M dwarf at $T = 3000$ K: $\lambda_\mathrm{max} \approx 966$ nm (near-infrared)
- Sun at $T = 5778$ K: $\lambda_\mathrm{max} \approx 501$ nm (green-yellow visible)
- O star at $T = 30000$ K: $\lambda_\mathrm{max} \approx 97$ nm (far-ultraviolet)

### 1.3 The Stefan–Boltzmann law

See: [Wikipedia — Stefan–Boltzmann law](https://en.wikipedia.org/wiki/Stefan%E2%80%93Boltzmann_law)

Integrating $B_\lambda$ over all wavelengths and all solid angles gives the total flux emitted per unit surface area:

$$F = \sigma_{\rm SB} T^4 \quad \text{[W m}^{-2}\text{]}$$

where $\sigma_{\rm SB} = 5.670\,374\,419 \times 10^{-8}$ W m$^{-2}$ K$^{-4}$ (NIST CODATA 2022; exact, derived from $h$, $c$, $k_B$).

For a spherical star of radius $R$ and uniform surface temperature $T$:

$$\boxed{L = 4\pi R^2 \sigma_{\rm SB} T^4}$$

### 1.4 Stellar spectral types

Stars are classified into spectral types O–B–A–F–G–K–M in order of decreasing temperature (mnemonic: *Oh Be A Fine Girl/Guy, Kiss Me*). Key values from Pecaut & Mamajek (2013), ApJS 208, 9:

| Spectral type | $T_\mathrm{eff}$ [K] | Colour | Example |
|:---:|:---:|:---:|:---:|
| O5 | 42 000 | blue-violet | Zeta Puppis |
| B0 | 30 000 | blue-white | Spica |
| A0 |  9 750 | white | Sirius A |
| F0 |  7 300 | yellow-white | Procyon A |
| G0 |  5 940 | yellow | Sun (G2) |
| K0 |  5 270 | orange | Epsilon Eridani |
| M0 |  3 850 | red-orange | Proxima Centauri |

### 1.5 Empirical stellar spectral libraries

Real stellar spectra differ from blackbodies because stellar atmospheres produce **absorption lines** — photons at specific energies are absorbed by atoms and molecules in the cooler outer layers. The **MILES** library (Sánchez-Blázquez et al. 2006, MNRAS 371, 703; Falcón-Barroso et al. 2011, A&A 532, A95) provides ~985 observed stellar spectra covering 3525–7500 Å at 2.5 Å FWHM resolution and is the standard reference for galaxy population synthesis.

Notable spectral features and the stellar types in which they are most prominent:

| Feature | Wavelength [Å] | Prominent in |
|---------|---------------|-------------|
| Balmer series (Hα, Hβ, Hγ, …) | 6563, 4861, 4341 | A and F stars (strongest at A0) |
| He I lines | 4026, 4471, 5876 | O and B stars |
| Ca II H&K | 3934, 3968 | G and K stars; key age indicator |
| G-band (CH) | 4304 | G and K giants |
| Mg b triplet | 5167–5183 | G/K stars; traces $\alpha$-element abundance |
| Na I D doublet | 5890, 5896 | K and M stars |
| TiO bands | 6190, 7060 | M dwarfs and giants |

### 1.6 Simple Stellar Populations and the Initial Mass Function

A **Simple Stellar Population (SSP)** assumes all stars formed in a single instantaneous burst at time $t = 0$ with identical chemical composition. At age $t$, all stars with main-sequence lifetimes $t_\mathrm{MS} < t$ have evolved off the main sequence; the **turnoff mass** $m_\mathrm{TO}(t)$ satisfies $t_\mathrm{MS}(m_\mathrm{TO}) = t$.

The **Initial Mass Function** (IMF) $\xi(m) = dN/d\log m$ describes the number of stars formed per unit logarithmic mass interval. Three widely used forms are:

$$\xi_\mathrm{Sal}(m) \propto m^{-2.35} \quad \text{(Salpeter 1955)}$$

$$\xi_\mathrm{Kro}(m) \propto \begin{cases} m^{-1.3} & m < 0.5\,M_\odot \\ m^{-2.3} & m > 0.5\,M_\odot \end{cases} \quad \text{(Kroupa 2001)}$$

$$\xi_\mathrm{Cha}(m) \propto \begin{cases} \exp\!\left[-\frac{(\log_{10} m - \log_{10} m_c)^2}{2\sigma^2}\right] & m \leq 1\,M_\odot \\ m^{-1.35} & m > 1\,M_\odot \end{cases} \quad \text{(Chabrier 2003, } m_c=0.079\,M_\odot,\,\sigma=0.69)$$

The integrated SSP spectrum at age $t$ is:

$$F_\lambda(t) = \int_{m_\mathrm{low}}^{m_\mathrm{TO}(t)} \xi(m)\, L(m)\, f_\lambda(m)\, dm$$

where $L(m)$ is the luminosity and $f_\lambda(m)$ the spectral shape — a Planck function in §8.4 or a real MILES spectrum in §8.5.

### 1.7 Observed galaxy spectra

Real galaxy spectra are the light-weighted sum of all their constituent stars. Two canonical types stand out in large spectroscopic surveys:

- **Star-forming / Emission-Line Galaxy (ELG):** Blue continuum dominated by massive OB stars. UV photons ionise surrounding gas, producing prominent nebular emission lines: **[O II] λ3727**, **Hβ λ4861**, **[O III] λλ4959, 5007**, **Hα λ6563**.

- **Quiescent / Luminous Red Galaxy (LRG):** Old stellar population with no recent star formation. Red continuum with a strong **4000 Å break** and deep absorption features: Ca H&K, G-band, Mg b, Na D.

The **4000 Å break** arises because metals in the atmospheres of cool stars absorb radiation shortward of 4000 Å, while young hot stars — which produce flux in the blue — are absent in passive populations. It is therefore one of the best single-wavelength age indicators in galaxy spectroscopy.

We use rest-frame stacked SDSS DR16 spectra from the `qmost_templates` repository (Comparat et al. 2020).

### 1.8 Stellar kinematics with ppxf

[`ppxf`](https://pypi.org/project/ppxf/) (Penalized Pixel-Fitting; Cappellari & Emsellem 2004, PASP 116, 138; Cappellari 2017, MNRAS 466, 798) fits an observed galaxy spectrum as the convolution of stellar templates with a **line-of-sight velocity distribution** (LOSVD):

$$F_\lambda^\mathrm{obs}(\lambda) = \int \mathrm{LOSVD}(v)\, F_\lambda^\mathrm{tpl}\!\left(\lambda\,e^{-v/c}\right) dv + P(\lambda)$$

The LOSVD is parameterised as a Gauss–Hermite series; the leading moments are the stellar **radial velocity** $V$ and **velocity dispersion** $\sigma$. A low-degree polynomial $P(\lambda)$ absorbs continuum shape differences between model and data. Fitting is done in log-wavelength space (uniform velocity steps) using penalised $\chi^2$ minimisation.

---

## References

- Carroll & Ostlie (2007), *An Introduction to Modern Astrophysics*, 2nd ed. — Sections 3.4–3.6
- NIST CODATA 2022 — physical constants ($h$, $c$, $k_B$ are exact since the 2019 SI redefinition): [physics.nist.gov/constants](https://physics.nist.gov/cuu/Constants/)
- Pecaut & Mamajek (2013), ApJS 208, 9 — spectral type effective temperatures: [arXiv:1307.2657](https://arxiv.org/abs/1307.2657)
- Sánchez-Blázquez et al. (2006), MNRAS 371, 703 — MILES stellar library: [arXiv:astro-ph/0604253](https://arxiv.org/abs/astro-ph/0604253)
- Falcón-Barroso et al. (2011), A&A 532, A95 — MILES library update: [DOI:10.1051/0004-6361/201116842](https://www.aanda.org/articles/aa/full_html/2011/08/aa16842-11/aa16842-11.html)
- Salpeter (1955), ApJ 121, 161 — Initial Mass Function
- Kroupa (2001), MNRAS 322, 231 — IMF: [arXiv:astro-ph/0009005](https://arxiv.org/abs/astro-ph/0009005)
- Chabrier (2003), PASP 115, 763 — IMF: [arXiv:astro-ph/0304382](https://arxiv.org/abs/astro-ph/0304382)
- Comparat et al. (2020), MNRAS 498, 1951 — qmost SED templates (SDSS DR16 stacks): [arXiv:2004.12244](https://arxiv.org/abs/2004.12244)
- Cappellari & Emsellem (2004), PASP 116, 138 — ppxf: [arXiv:astro-ph/0311619](https://arxiv.org/abs/astro-ph/0311619)
- Cappellari (2017), MNRAS 466, 798 — ppxf update: [arXiv:1607.08538](https://arxiv.org/abs/1607.08538)

---

## 2. Setup and imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import AutoMinorLocator
from scipy.integrate import quad
import warnings
warnings.filterwarnings('ignore')

# ── Physical constants (NIST CODATA 2022; h, c, k_B are exact since 2019 SI) ─
h        = 6.62607015e-34     # Planck constant [J s]
c        = 2.99792458e8       # speed of light [m/s]
k_B      = 1.380649e-23       # Boltzmann constant [J/K]
sigma_SB = 5.670374419e-8     # Stefan-Boltzmann constant [W m^-2 K^-4]
b_Wien   = 2.897771955e-3     # Wien displacement constant [m K]

# ── IAU nominal solar constants ───────────────────────────────────────────
R_sun  = 6.957e8              # solar radius [m]
T_sun  = 5778.0               # solar effective temperature [K]
L_sun  = 3.828e26             # solar luminosity [W]
M_sun  = 1.989e30             # solar mass [kg]

print('Physical constants loaded (NIST CODATA 2022).')
print(f'  h       = {h:.8e} J s  (exact)')
print(f'  c       = {c:.8e} m/s  (exact)')
print(f'  k_B     = {k_B:.8e} J/K  (exact)')
print(f'  sigma   = {sigma_SB:.9e} W m^-2 K^-4')
print(f'  b_Wien  = {b_Wien:.9e} m K')
print()
print('IAU nominal solar constants:')
print(f'  R_sun   = {R_sun:.3e} m')
print(f'  T_sun   = {T_sun:.0f} K')
print(f'  L_sun   = {L_sun:.3e} W')

# ── Plot style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'lines.linewidth': 2.0,
})

# ── Spectral type data (Pecaut & Mamajek 2013, ApJS 208, 9) ───────────────
SPTYPES = {
    # name : (T_eff [K], B-V_intrinsic, MS_radius [R_sun])
    'O5':  (42000, -0.33, 12.0),
    'B0':  (30000, -0.30,  7.4),
    'A0':  ( 9750,  0.00,  2.4),
    'F0':  ( 7300,  0.30,  1.5),
    'G0':  ( 5940,  0.58,  1.1),
    'K0':  ( 5270,  0.81,  0.85),
    'M0':  ( 3850,  1.40,  0.60),
}

print('\nSpectral type reference values loaded (Pecaut & Mamajek 2013).')

---

## 3. The Planck function: $B_\lambda$ and $B_\nu$

We define the two forms of the Planck function and verify they are related by the Jacobian $B_\lambda = B_\nu \cdot c/\lambda^2$.

In [ ]:
def B_lambda(lam, T):
    """
    Planck spectral radiance per unit wavelength.

    Parameters
    ----------
    lam : array-like, float
        Wavelength [m]
    T : float
        Temperature [K]

    Returns
    -------
    B : array-like
        Spectral radiance [W m^-2 m^-1 sr^-1]
    """
    lam = np.asarray(lam, dtype=float)
    exponent = h * c / (lam * k_B * T)
    # Guard against overflow for very short wavelengths / low temperatures
    exponent = np.minimum(exponent, 700.0)
    return (2.0 * h * c**2 / lam**5) / (np.exp(exponent) - 1.0)


def B_nu(nu, T):
    """
    Planck spectral radiance per unit frequency.

    Parameters
    ----------
    nu : array-like, float
        Frequency [Hz]
    T : float
        Temperature [K]

    Returns
    -------
    B : array-like
        Spectral radiance [W m^-2 Hz^-1 sr^-1]
    """
    nu = np.asarray(nu, dtype=float)
    exponent = h * nu / (k_B * T)
    exponent = np.minimum(exponent, 700.0)
    return (2.0 * h * nu**3 / c**2) / (np.exp(exponent) - 1.0)


# ── Verify the Jacobian relation B_lambda = B_nu * (c / lambda^2) ──────────
lam_test = 500e-9   # 500 nm
nu_test  = c / lam_test

val_lam = B_lambda(lam_test, T_sun)
val_nu  = B_nu(nu_test, T_sun)
val_converted = val_nu * c / lam_test**2   # B_nu -> B_lambda

print('Verification of Jacobian B_lambda = B_nu * c / lambda^2')
print(f'  B_lambda(500 nm, T_sun)           = {val_lam:.4e} W m^-3 sr^-1')
print(f'  B_nu(nu_500nm, T_sun) * c/lam^2   = {val_converted:.4e} W m^-3 sr^-1')
print(f'  Relative difference: {abs(val_lam - val_converted)/val_lam * 100:.6f}%')

---

## 4. Plotting blackbody spectra for a range of stellar temperatures

We plot $B_\lambda(T)$ for four representative temperatures:
- $T = 3000$ K — cool M dwarf
- $T = 5778$ K — the Sun (G2 star)
- $T = 10000$ K — A-type star (Sirius-like)
- $T = 30000$ K — hot B-type star

The **visible spectrum** (380–700 nm) is indicated as a shaded band.

In [ ]:
# Wavelength grid: 100 nm to 3000 nm
lam_nm  = np.linspace(100, 3000, 5000)      # [nm]
lam_m   = lam_nm * 1e-9                     # [m]

# Temperatures to plot
temps   = [3000, 5778, 10000, 30000]
colors  = ['firebrick', 'goldenrod', 'steelblue', 'mediumpurple']
labels  = ['3 000 K (M dwarf)', '5 778 K (Sun, G2)', '10 000 K (A star)', '30 000 K (B star)']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, (ax, norm) in enumerate(zip(axes, [False, True])):

    # Shade the visible spectrum (380–700 nm)
    ax.axvspan(380, 700, color='lightyellow', alpha=0.8, zorder=0, label='Visible (380–700 nm)')

    for T, col, lbl in zip(temps, colors, labels):
        B = B_lambda(lam_m, T) * 1e-9   # convert to W m^-2 nm^-1 sr^-1

        if norm:
            B = B / B.max()   # normalise to peak = 1

        ax.plot(lam_nm, B, color=col, lw=2, label=lbl)

        # Mark Wien peak
        lam_peak_nm = b_Wien / T * 1e9
        if 100 < lam_peak_nm < 3000:
            B_peak = B_lambda(lam_peak_nm * 1e-9, T) * 1e-9
            if norm:
                B_peak = 1.0
            ax.axvline(lam_peak_nm, color=col, lw=0.8, ls='--', alpha=0.7)
            if ax_idx == 1:  # only annotate on right panel
                ax.annotate(f'{lam_peak_nm:.0f} nm',
                            xy=(lam_peak_nm, B_peak),
                            xytext=(lam_peak_nm + 80, B_peak - 0.05),
                            fontsize=8, color=col,
                            arrowprops=dict(arrowstyle='->', color=col, lw=0.8))

    ax.set_xlabel('Wavelength $\\lambda$ [nm]')
    if norm:
        ax.set_ylabel('Normalised spectral radiance')
        ax.set_title('Normalised Planck spectra (peak = 1)')
    else:
        ax.set_ylabel('$B_\\lambda$ [W m$^{-2}$ nm$^{-1}$ sr$^{-1}$]')
        ax.set_title('Planck blackbody spectra')
    ax.set_xlim(100, 2500)
    ax.legend(fontsize=9, loc='upper right')
    ax.xaxis.set_minor_locator(AutoMinorLocator())

plt.tight_layout()
plt.savefig('blackbody_spectra.png', dpi=150, bbox_inches='tight')
plt.show()
print('Left: absolute spectra — note 30 000 K peak is off the plot (in UV).')
print('Right: normalised spectra — peak wavelength shifts clearly with temperature.')

---

## 5. Wien's displacement law

The Wien peak $\lambda_\mathrm{max} = b_W / T$ shifts to shorter wavelengths as temperature increases. Let us plot this over the full range of stellar temperatures and label the spectral types.

In [ ]:
# Temperature grid for Wien's law
T_grid = np.logspace(np.log10(2000), np.log10(50000), 300)
lam_wien_nm = b_Wien / T_grid * 1e9   # [nm]

fig, ax = plt.subplots(figsize=(9, 5))

# Wien's law curve
ax.loglog(T_grid, lam_wien_nm, 'k-', lw=2.5, label="Wien's law: $\\lambda_\\mathrm{max} = b_W / T$")

# Shade spectral regions
regions = [
    (100, 380,  'ultraviolet', '#d0a0ff', 0.3),
    (380, 700,  'visible',     'lightyellow', 0.8),
    (700, 3000, 'near-IR',     '#ffc080', 0.3),
]
for lam_lo, lam_hi, name, col, alpha in regions:
    ax.axhspan(lam_lo, lam_hi, color=col, alpha=alpha, zorder=0)
    ax.text(50000, (lam_lo * lam_hi)**0.5, name, ha='right', va='center',
            fontsize=9, color='gray', style='italic')

# Mark spectral types
cmap_spt = plt.cm.coolwarm_r
for i, (spt, (Teff, BV, R_R)) in enumerate(SPTYPES.items()):
    lam_peak = b_Wien / Teff * 1e9
    col_pt = cmap_spt(i / (len(SPTYPES) - 1))
    ax.scatter(Teff, lam_peak, color=col_pt, s=100, zorder=10, edgecolors='k', lw=0.8)
    ax.annotate(spt, xy=(Teff, lam_peak),
                xytext=(Teff * 1.12, lam_peak * 1.05),
                fontsize=9, ha='left',
                arrowprops=dict(arrowstyle='->', lw=0.6, color='gray'))

ax.set_xlabel('Effective temperature $T_\\mathrm{eff}$ [K]')
ax.set_ylabel('Peak wavelength $\\lambda_\\mathrm{max}$ [nm]')
ax.set_title("Wien's displacement law: $\\lambda_\\mathrm{max} \\cdot T = b_W = 2.898 \\times 10^{-3}$ m K")
ax.legend()
ax.set_xlim(2000, 60000)
ax.set_ylim(50, 2000)
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
plt.tight_layout()
plt.savefig('wien_displacement.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nWien peak wavelengths for each spectral type:")
print(f"{'Spt':>4}  {'T_eff [K]':>10}  {'lambda_max [nm]':>16}  {'Region':>12}")
for spt, (Teff, BV, R_R) in SPTYPES.items():
    lam_p = b_Wien / Teff * 1e9
    region = 'UV' if lam_p < 380 else ('Visible' if lam_p < 700 else 'Near-IR')
    print(f"{spt:>4}  {Teff:>10.0f}  {lam_p:>16.1f}  {region:>12}")

---

## 6. Stefan–Boltzmann law: total luminosity

The total luminosity of a star with radius $R$ and effective temperature $T$ is:

$$L = 4\pi R^2 \sigma_{\rm SB} T^4$$

To compute $L$ for each spectral type, we need both $T_\mathrm{eff}$ and the **main-sequence radius** $R$. We use tabulated values from Pecaut & Mamajek (2013).

In [ ]:
def stefan_boltzmann_luminosity(R_Rsun, T_K):
    """
    Compute stellar luminosity using the Stefan-Boltzmann law.

    Parameters
    ----------
    R_Rsun : float
        Stellar radius in solar units
    T_K : float
        Effective temperature in Kelvin

    Returns
    -------
    L_W : float
        Luminosity in Watts
    L_Lsun : float
        Luminosity in solar units
    """
    R_m    = R_Rsun * R_sun
    L_W    = 4.0 * np.pi * R_m**2 * sigma_SB * T_K**4
    L_Lsun = L_W / L_sun
    return L_W, L_Lsun


# Compute L for each spectral type and for a range of radii/temperatures
print('Luminosity for each spectral type (main-sequence radii from Pecaut & Mamajek 2013):')
print(f'\n{"Spt":>4}  {"T_eff [K]":>10}  {"R [R_sun]":>10}  {"L [W]":>12}  {"L/L_sun":>10}  {"lambda_max [nm]":>16}')
print('-' * 75)

luminosities = {}
for spt, (Teff, BV, R_R) in SPTYPES.items():
    L_W, L_ratio = stefan_boltzmann_luminosity(R_R, Teff)
    lam_p = b_Wien / Teff * 1e9
    luminosities[spt] = (Teff, R_R, L_W, L_ratio)
    print(f"{spt:>4}  {Teff:>10.0f}  {R_R:>10.2f}  {L_W:>12.3e}  {L_ratio:>10.3f}  {lam_p:>16.1f}")

print(f"\nSun:  T={T_sun:.0f} K, R=1.00 R_sun, L={L_sun:.3e} W, L/L_sun=1.000")
print(f"  (S-B formula gives: {stefan_boltzmann_luminosity(1.0, T_sun)[0]:.3e} W)")

# Show L ∝ T^4 scaling for fixed radius
print('\nIllustration: doubling T increases L by factor 2^4 = 16')
T_ref = 5000.0
L_ref, _ = stefan_boltzmann_luminosity(1.0, T_ref)
for fac in [0.5, 1.0, 2.0, 3.0]:
    L_fac, _ = stefan_boltzmann_luminosity(1.0, T_ref * fac)
    print(f'  T = {T_ref*fac:.0f} K  =>  L/L_ref = {L_fac/L_ref:.1f}')

In [ ]:
# Plot: L/Lsun vs T_eff for fixed radius (R = 1 R_sun) to illustrate T^4 law
# and for the spectral-type data points
T_arr = np.linspace(2500, 50000, 500)
L_fixed_R, _ = stefan_boltzmann_luminosity(np.ones_like(T_arr), T_arr)
L_fixed_R_norm = L_fixed_R / L_sun

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left panel: L/Lsun vs T for fixed R=1 Rsun
ax = axes[0]
ax.plot(T_arr, L_fixed_R_norm, 'darkblue', lw=2.5,
        label='$R = 1\\,R_\\odot$ (Stefan–Boltzmann $L \\propto T^4$)')
ax.scatter([T_sun], [1.0], color='gold', s=200, zorder=10,
           edgecolors='k', lw=0.8, label='Sun')
ax.set_xlabel('Effective temperature $T_\\mathrm{eff}$ [K]')
ax.set_ylabel('Luminosity $L/L_\\odot$')
ax.set_yscale('log')
ax.set_title('Stefan–Boltzmann: $L = 4\\pi R^2 \\sigma T^4$  (fixed $R = R_\\odot$)')
ax.legend()
ax.xaxis.set_minor_locator(AutoMinorLocator())

# Right panel: L vs T for the spectral types (with their MS radii)
ax2 = axes[1]
cmap_spt = plt.cm.RdYlBu_r
for i, (spt, (Teff, R_R, L_W, L_ratio)) in enumerate(luminosities.items()):
    col = cmap_spt(i / (len(luminosities) - 1))
    ax2.scatter(Teff, L_ratio, color=col, s=150, zorder=10,
                edgecolors='k', lw=0.8, label=f'{spt}  ({Teff:,} K)')
    ax2.annotate(spt, xy=(Teff, L_ratio),
                 xytext=(Teff * 1.05, L_ratio * 1.3),
                 fontsize=9, arrowprops=dict(arrowstyle='->', lw=0.6))

# Approximate ZAMS line for reference: L ~ (T/T_sun)^8 * (R/R_sun)^2 is not
# exact; instead just connect the spectral-type points as a guide.
T_spt = np.array([v[0] for v in luminosities.values()])
L_spt = np.array([v[3] for v in luminosities.values()])
idx   = np.argsort(T_spt)
ax2.plot(T_spt[idx], L_spt[idx], 'k--', lw=1, alpha=0.5, label='ZAMS (Pecaut & Mamajek 2013)')
ax2.scatter([T_sun], [1.0], color='gold', s=200, zorder=10,
            edgecolors='k', lw=0.8)
ax2.set_xlabel('Effective temperature $T_\\mathrm{eff}$ [K]')
ax2.set_ylabel('Luminosity $L/L_\\odot$')
ax2.set_yscale('log')
ax2.set_xscale('log')
ax2.invert_xaxis()
ax2.set_title('Spectral-type luminosities (MS radii, Stefan–Boltzmann)')
ax2.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.savefig('stefan_boltzmann_luminosity.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 7. Observed Stellar Spectra: Empirical Spectral Libraries

Real stellar spectra differ from perfect blackbodies because stellar atmospheres produce **absorption lines** and **molecular bands** imprinted on the continuum — these arise from bound-bound atomic and molecular transitions in the photosphere.

Two major empirical libraries provide observed templates across the full spectral-type sequence:

### 7.1 The MILES Stellar Library

The **MILES** (Medium resolution INT Library of Empirical Spectra) library contains ~1000 stars spanning the full H-R diagram at medium spectral resolution.

- **Coverage:** 3525–7500 Å at 2.5 Å FWHM spectral resolution
- **Website:** [research.iac.es/proyecto/miles](https://research.iac.es/proyecto/miles/pages/stellar-libraries/miles-library.php)
- **Latest release:** Falcón-Barroso et al. (2011), A&A 532, A95 — [DOI:10.1051/0004-6361/201116842](https://www.aanda.org/articles/aa/full_html/2011/08/aa16842-11/aa16842-11.html)

Each MILES spectrum is a 1D FITS file with flux density in erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$ on a uniform grid (3525–7500 Å, step = 0.9 Å, $N_\lambda = 4374$ pixels).

### 7.2 The MaStar Stellar Library (SDSS-IV)

The **MaStar** (MaNGA Stellar Library) is one of the largest empirical stellar libraries, with ~59 000 unique stellar spectra covering a wide range of stellar parameters.

- **Coverage:** 3622–10 354 Å at resolving power $R \approx 1800$
- **Website:** [sdss4.org/surveys/mastar](https://www.sdss4.org/surveys/mastar/)
- **Reference:** Yan et al. (2019), ApJ 883, 175 — [arXiv:1908.05537](https://arxiv.org/abs/1908.05537)
- **Data release:** SDSS DR17 (final release, 2022)

MaStar covers $3000 < T_\mathrm{eff} < 35\,000$ K and $-2 < [\mathrm{Fe/H}] < +0.5$, making it ideal for stellar population synthesis.

### 7.3 Dominant spectral features by stellar type

| Type | $T_\mathrm{eff}$ [K] | Dominant features |
|:----:|:---:|:---|
| O | >30 000 | He II; H Balmer series (weak at highest $T$) |
| B | 10 000–30 000 | He I; strong H Balmer series |
| A | 7 500–10 000 | Maximum H Balmer absorption; Ca II K (weak) |
| F | 6 000–7 500 | Ca II H&K; G-band (CH); H Balmer weakening |
| G | 5 200–6 000 | Ca II H&K strong; G-band; many Fe lines |
| K | 3 700–5 200 | Ca II very strong; Na I D doublet; TiO bands begin |
| M | <3 700 | TiO and VO molecular bands dominate red/NIR |

In [ ]:
import os, io, tarfile, urllib.request, ssl
from astropy.io import fits

# ── SSL context required for the IAC server (DH key workaround) ───────────
CTX = ssl.create_default_context()
CTX.set_ciphers('DEFAULT:@SECLEVEL=1')

MILES_TAR_URL = ('https://research.iac.es/proyecto/miles/media/'
                 'tarfiles/Stellar_Libraries/MILES_library_v9.1_FITS.tar.gz')
MILES_CAT_URL = 'https://cdsarc.cds.unistra.fr/ftp/J/MNRAS/371/703/catalog.dat'
MILES_TAR_LOCAL = 'MILES_library_v9.1_FITS.tar.gz'

# ── 1. Download and parse the MILES catalog from CDS (96 kB) ─────────────
print('Downloading MILES catalog from CDS…')
raw = urllib.request.urlopen(MILES_CAT_URL, context=CTX, timeout=30).read()
cat_lines = raw.decode().split('\n')

# Fixed-width format: FileName[0:10], SpType[58:72], Teff[73:78]
miles_catalog = {}
for line in cat_lines:
    if len(line) < 78:
        continue
    fname  = line[0:10].strip()
    name   = line[11:26].strip()
    sptype = line[58:72].strip()
    teff_s = line[73:78].strip()
    teff   = int(teff_s) if teff_s.lstrip('-').isdigit() else -1
    if fname and sptype and teff > 0:
        miles_catalog[fname] = {'name': name, 'sptype': sptype, 'teff': teff}

print(f'  {len(miles_catalog)} stars in MILES catalog.')

# ── 2. Chosen representative stars (one per spectral class) ──────────────
# Selected from the catalog for broad Teff coverage and known spectral type.
# Arcturus (K2IIIp) is a famous stellar-spectroscopy calibrator.
CHOSEN = {
    'B': ('s0011.fits', 'B2 IV',  21581),   # HD 886
    'A': ('s0006.fits', 'A1 V',    8140),   # HD 884 region
    'F': ('s0004.fits', 'F0',      6380),   # HD 4
    'G': ('s0001.fits', 'G3 V',    5305),   # HD 224930
    'K': ('s0501.fits', 'K2 IIIp', 4361),   # HD 124897 = Arcturus
    'M': ('s0012.fits', 'M6 V',    3330),   # HD 1326B
}

print('\nSelected representative stars:')
for sp, (fname, sptype, teff) in CHOSEN.items():
    row = miles_catalog.get(fname, {})
    print(f'  {sp}: {fname}  name={row.get("name","?"):16s}  '
          f'SpType={row.get("sptype", sptype):14s}  Teff={teff} K')

# ── 3. Download MILES FITS library (14.8 MB) — cached after first run ─────
if not os.path.exists(MILES_TAR_LOCAL):
    print(f'\nDownloading MILES FITS library (~14.8 MB) from IAC…')
    with urllib.request.urlopen(MILES_TAR_URL, context=CTX, timeout=300) as resp:
        total = int(resp.headers.get('Content-Length', 0))
        buf, done = bytearray(), 0
        while True:
            chunk = resp.read(131072)
            if not chunk:
                break
            buf += chunk
            done += len(chunk)
            if total:
                print(f'\r  {done/1e6:.1f}/{total/1e6:.1f} MB', end='', flush=True)
    with open(MILES_TAR_LOCAL, 'wb') as f:
        f.write(buf)
    print(f'\n  Saved → {MILES_TAR_LOCAL}')
else:
    print(f'\nUsing cached {MILES_TAR_LOCAL}  ({os.path.getsize(MILES_TAR_LOCAL)/1e6:.1f} MB)')

In [ ]:
import pyneb as pn

# ── Build stellar line catalogue ──────────────────────────────────────────
# Balmer series: pn.RecAtom('H', 1).wave_Ang is a (40×40) numpy array.
# wave_Ang[1, j] = wavelength [Å] of transition n=(j+1)→n=2 (Balmer series).
h1 = pn.RecAtom('H', 1)

BALMER_DEF = {2: 'Hα', 3: 'Hβ', 4: 'Hγ', 5: 'Hδ', 6: 'Hε', 7: 'H8', 8: 'H9'}
balmer_lines = [
    (h1.wave_Ang[1, j] / 10.0, name, 'crimson', '--')
    for j, name in BALMER_DEF.items()
    if 350 < h1.wave_Ang[1, j] / 10.0 < 743
]

# He I lines: pn.RecAtom('He', 1).wave_Ang is None → hardcoded vacuum wavelengths
he1_lines = [
    (402.6, 'He I',  'blueviolet', ':'),   # 4026 Å
    (438.8, 'He I',  'blueviolet', ':'),   # 4388 Å
    (447.1, 'He I',  'blueviolet', ':'),   # 4471 Å (strong in B/A stars)
    (471.3, 'He I',  'blueviolet', ':'),   # 4713 Å
    (501.6, 'He I',  'blueviolet', ':'),   # 5016 Å
    (587.6, 'He D3', 'blueviolet', ':'),   # 5876 Å (near Na I D)
    (667.8, 'He I',  'blueviolet', ':'),   # 6678 Å
    (706.5, 'He I',  'blueviolet', ':'),   # 7065 Å
]

# Metal lines and molecular bands
metal_lines = [
    (393.4, 'Ca K',   'steelblue',   '-'),   # Ca II K resonance
    (396.8, 'Ca H',   'steelblue',   '-'),   # Ca II H (blends with Hε)
    (422.7, 'Ca I',   'royalblue',   '-'),   # Ca I 4227
    (430.8, 'G-band', 'sienna',      '-'),   # CH G-band
    (516.7, 'Mg b',   'forestgreen', '-'),   # Mg I b triplet centroid
    (589.3, 'Na D',   'darkorange',  '-'),   # Na I D doublet centroid
    (619.2, 'TiO',    'firebrick',   '-.'),  # TiO γ band head
    (706.0, 'TiO',    'firebrick',   '-.'),  # TiO α band head
]

STELLAR_LINES = sorted(balmer_lines + he1_lines + metal_lines, key=lambda x: x[0])
print(f'{len(STELLAR_LINES)} stellar lines loaded '
      f'({len(balmer_lines)} Balmer from pyNEB,  '
      f'{len(he1_lines)} He I,  {len(metal_lines)} metal/mol.)')

# ── 4. Extract selected MILES spectra ─────────────────────────────────────
# Wavelength grid: CRVAL1=3500 Å, CDELT1=0.9 Å, NAXIS1=4367 → 350–743 nm

def read_miles_from_tar(tf, filename):
    """Extract a single MILES FITS spectrum from an open tarfile."""
    member = tf.getmember(filename)
    raw    = tf.extractfile(member).read()
    with fits.open(io.BytesIO(raw)) as hdul:
        hdr    = hdul[0].header
        flux   = hdul[0].data.flatten().astype(float)   # erg/s/cm²/Å
        crval1 = float(hdr['CRVAL1'])
        cdelt1 = float(hdr['CDELT1'])
        lam_nm = (crval1 + cdelt1 * np.arange(len(flux))) / 10.0   # Å → nm
    return lam_nm, flux

spt_colors = {'B': 'mediumpurple', 'A': 'steelblue', 'F': 'limegreen',
              'G': 'goldenrod',     'K': 'darkorange', 'M': 'firebrick'}
spectra_obs = {}

with tarfile.open(MILES_TAR_LOCAL, 'r:gz') as tf:
    for sp, (fname, sptype, teff) in CHOSEN.items():
        try:
            lam_nm, flux = read_miles_from_tar(tf, fname)
            spectra_obs[sp] = (lam_nm, flux, sptype, teff)
            print(f'  {sp}: {fname}  ({sptype}, Teff={teff} K)  '
                  f'flux = [{flux.min():.2e}, {flux.max():.2e}] erg/s/cm²/Å')
        except Exception as e:
            print(f'  {sp}: FAILED — {e}')

# ── 5. Plot: observed spectra vs Planck + line labels on every panel ───────
Y_ALT = [1.30, 1.17]   # alternating label heights (plot_full.py style)

fig, axes = plt.subplots(len(spectra_obs), 1, figsize=(13, 17), sharex=True)
ax_list   = list(axes)

for ax, (sp, (lam_nm, flux, sptype, teff)) in zip(ax_list, spectra_obs.items()):
    col = spt_colors[sp]

    # Normalise to peak in the 400–700 nm window
    mask_vis = (lam_nm >= 400) & (lam_nm <= 700)
    norm_val  = flux[mask_vis].max() if mask_vis.any() else flux.max()
    flux_norm = flux / norm_val

    # Planck blackbody at same Teff, same normalisation
    B      = B_lambda(lam_nm * 1e-9, teff)
    B_norm = B / B[mask_vis].max() if mask_vis.any() else B / B.max()

    ax.axvspan(380, 700, color='lightyellow', alpha=0.5, zorder=0)
    ax.plot(lam_nm, B_norm,    color=col, lw=1.0, ls='--', alpha=0.5,
            label=f'Planck  T={teff} K')
    ax.plot(lam_nm, flux_norm, color=col, lw=1.3,
            label=f'MILES {sp}: {sptype}  '
                  f'({miles_catalog.get(CHOSEN[sp][0], {}).get("name","?")})  Teff={teff} K')

    # ── Spectral line labels: every panel, alternating heights ────────────
    for idx, (lam_l, label, lc, ls) in enumerate(STELLAR_LINES):
        if 350 < lam_l < 743:
            ax.axvline(lam_l, color=lc, lw=0.6, ls=ls, alpha=0.65, zorder=2)
            ax.text(lam_l, Y_ALT[idx % 2], label,
                    rotation=90, ha='center', va='bottom',
                    fontsize=6, color=lc, alpha=0.85)

    ax.set_ylim(-0.15, 1.45)
    ax.set_ylabel('Norm. flux', fontsize=8)
    ax.legend(loc='lower right', fontsize=8.5, framealpha=0.85)
    ax.tick_params(labelsize=8)

axes[-1].set_xlabel('Wavelength $\\lambda$ [nm]', fontsize=12)
axes[-1].set_xlim(350, 743)

fig.suptitle(
    'MILES library: observed stellar spectra (solid) vs Planck blackbody (dashed)\n'
    'Balmer = crimson  |  He I = violet  |  Ca = blue  |  metals = green/orange  |  TiO = red\n'
    'Sánchez-Blázquez et al. (2006), MNRAS 371, 703',
    fontsize=10, y=0.999)
plt.tight_layout(rect=[0, 0, 1, 0.998])
plt.savefig('miles_spectral_sequence.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nNote: the deviation of the solid line from the dashed line reveals:')
print('  B: weaker visible curvature (peak in UV); He I absorption features')
print('  A: maximum Balmer absorption; Balmer series dominates')
print('  F/G: Ca II H&K and G-band appear; Balmer weakening')
print('  K: Ca II very strong; Na D prominent; line blanketing reddens continuum')
print('  M: TiO molecular bands create broad, deep troughs in red/NIR')

---

## 8. Stellar Population Synthesis

A galaxy — or a star cluster — contains a large number of stars with different masses, temperatures, and luminosities. The observed spectrum of such a system is the **sum of the spectra of all constituent stars**, weighted by their relative contributions.

**Stellar population synthesis (SPS)** is the technique of predicting this integrated spectrum from a model of the underlying stellar population.

### 8.1 The Initial Mass Function (IMF)

Stars do not form with equal probability at all masses. The **IMF** $\xi(m)$ describes the number of stars formed per unit logarithmic mass interval:

$$\xi(m) = \frac{dN}{d\log m}$$

Three widely used forms:

**Salpeter (1955):** A single power law
$$\xi_\mathrm{Sal}(m) \propto m^{-2.35}$$

**Kroupa (2001):** A broken power law with a flatter slope at low masses
$$\xi_\mathrm{Kro}(m) \propto \begin{cases} m^{-1.3} & 0.08 < m/M_\odot \leq 0.5 \\ m^{-2.3} & m/M_\odot > 0.5 \end{cases}$$

**Chabrier (2003):** A log-normal distribution below $1\,M_\odot$ joined to a Salpeter power law above (Chabrier 2003, PASP 115, 763; [arXiv:astro-ph/0304382](https://arxiv.org/abs/astro-ph/0304382)):

$$\xi_\mathrm{Cha}(m) \propto \begin{cases}
\exp\!\left[-\dfrac{(\log_{10} m - \log_{10} m_c)^2}{2\sigma^2}\right] & m \leq 1\,M_\odot \\[6pt]
m^{-1.35} & m > 1\,M_\odot
\end{cases}$$

with characteristic mass $m_c = 0.079\,M_\odot$ and width $\sigma = 0.69\,\mathrm{dex}$. The Chabrier IMF is continuous at $1\,M_\odot$ and predicts fewer low-mass stars than Salpeter, making it the current standard for galaxy SPS modelling.

### 8.2 Simple Stellar Population (SSP)

An **SSP** assumes all stars formed in a single instantaneous burst at time $t = 0$ with the same chemical composition. At a later time $t$:
- Stars with main-sequence lifetime $t_\mathrm{MS} > t$ are still on the main sequence
- Stars with $t_\mathrm{MS} < t$ have evolved off (become giants, white dwarfs, …)

The turnoff mass $m_\mathrm{TO}(t)$ is defined by $t_\mathrm{MS}(m_\mathrm{TO}) = t$.

**Main-sequence lifetimes** have been computed from detailed stellar evolution grids; a seminal reference is Maeder & Meynet (1989, A&A 210, 155), who published evolutionary tracks from $0.85$ to $120\,M_\odot$. A useful order-of-magnitude approximation valid for solar-type stars is:

$$t_\mathrm{MS} \approx 10\,\mathrm{Gyr} \times \left(\frac{m}{M_\odot}\right)^{-2.5}$$

This approximation works well for $0.5 \lesssim m/M_\odot \lesssim 10$ but **underestimates** the lifetime of very massive stars ($m \gtrsim 20\,M_\odot$), for which radiation pressure reduces the effective luminosity exponent and stellar-wind mass loss becomes important (for $m = 60\,M_\odot$, the formula gives $\sim 0.4$ Myr whereas models give $\sim 4$ Myr).

| $m\,[M_\odot]$ | $t_\mathrm{MS}$ formula [Gyr] | $t_\mathrm{MS}$ Maeder & Meynet (1989) [Gyr] | Approx. type |
|:---:|:---:|:---:|:---:|
| 0.8 | 17.5 | ~18 | K |
| 1.0 | 10.0 | ~10 | G |
| 1.5 | 3.6 | ~2.5 | F |
| 2.0 | 1.8 | ~1.3 | A |
| 3.0 | 0.64 | ~0.42 | B |
| 5.0 | 0.18 | ~0.098 | B |
| 10 | 0.032 | ~0.022 | B |
| 20 | 0.0056 | ~0.0077 | O |
| 40 | 0.0010 | ~0.0048 | O |
| 60 | 0.00036 | ~0.0042 | O |

### 8.3 The integrated spectrum

The total flux from an SSP at age $t$ is:

$$F_\lambda(t) = \int_{m_\mathrm{low}}^{m_\mathrm{TO}(t)} \xi(m) \, L(m) \, f_\lambda(m) \, dm$$

where $m_\mathrm{TO}(t)$ is the main-sequence turnoff mass, $L(m)$ the luminosity, and $f_\lambda(m)$ the normalised spectral shape — a Planck function in §8.4 or real MILES spectra in §8.5.

In [ ]:
# ── IMF definitions ─────────────────────────────────────────────────────────

def imf_salpeter(m):
    """Salpeter (1955) IMF: dN/dlog(m) ∝ m^(-2.35)."""
    return np.asarray(m, dtype=float)**(-2.35)

def imf_kroupa(m):
    """Kroupa (2001) IMF: broken power law."""
    m = np.asarray(m, dtype=float)
    return np.where(m < 0.5, m**(-1.3), m**(-2.3))

def imf_chabrier(m):
    """Chabrier (2003) IMF: log-normal below 1 M☉ + Salpeter above.
    PASP 115, 763; arXiv:astro-ph/0304382. Parameters: m_c=0.079 M☉, σ=0.69.
    """
    m = np.asarray(m, dtype=float)
    log_mc  = np.log10(0.079)   # characteristic mass [M_sun]
    sigma   = 0.69               # dispersion [dex]
    A       = 0.158              # lognormal normalisation
    xi_at_1 = A * np.exp(-((0.0 - log_mc)**2) / (2 * sigma**2))  # value at m=1 (continuity)
    return np.where(
        m < 1.0,
        A * np.exp(-((np.log10(m) - log_mc)**2) / (2 * sigma**2)),
        xi_at_1 * m**(-1.35)
    )

# ── Main-sequence scaling relations ─────────────────────────────────────────

def ms_temperature(m):
    """Approximate T_eff [K] for main-sequence star of mass m [M_sun]."""
    m = np.asarray(m, dtype=float)
    return T_sun * np.where(m > 2.0, m**0.65, m**0.50)

def ms_radius(m):
    """Approximate stellar radius [R_sun] for main-sequence star of mass m."""
    return np.asarray(m, dtype=float)**0.80

def ms_lifetime(m):
    """Approximate MS lifetime [Gyr]: t ≈ 10 × (m/M_sun)^-2.5 Gyr.
    Calibrated at 1 M_sun; underestimates lifetimes for m > 20 M_sun
    (see Maeder & Meynet 1989, A&A 210, 155).
    """
    return 10.0 * np.asarray(m, dtype=float)**(-2.5)

# ── IMF comparison plot ─────────────────────────────────────────────────────
m_grid = np.logspace(np.log10(0.1), np.log10(100), 300)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
for fn, name, col, ls in [
    (imf_salpeter, 'Salpeter (1955)', 'steelblue',   '-'),
    (imf_kroupa,   'Kroupa (2001)',   'firebrick',   '--'),
    (imf_chabrier, 'Chabrier (2003)', 'forestgreen', '-.'),
]:
    xi = fn(m_grid)
    xi /= xi[m_grid >= 1.0][0]   # normalise to 1 at 1 M_sun
    ax.loglog(m_grid, xi, color=col, ls=ls, lw=2, label=name)

ax.axvline(0.5,   color='gray',        lw=0.8, ls=':', label='Kroupa break (0.5 M☉)')
ax.axvline(0.079, color='forestgreen', lw=0.8, ls=':', label='Chabrier $m_c$ (0.079 M☉)')
ax.set_xlabel('Stellar mass $m$ [$M_\\odot$]')
ax.set_ylabel('$\\xi(m) = dN/d\\log m$  (normalised at 1 $M_\\odot$)')
ax.set_title('Initial Mass Functions')
ax.legend(fontsize=9)
ax.set_xlim(0.1, 100)

# ── MS scaling relations ────────────────────────────────────────────────────
ax2 = axes[1]
T_ms = ms_temperature(m_grid)
R_ms = ms_radius(m_grid)
L_ms = (R_ms**2) * (T_ms / T_sun)**4

ax2.loglog(m_grid, L_ms,               'k-',           lw=2, label='$L/L_\\odot$')
ax2.loglog(m_grid, T_ms / T_sun,       'darkorange',   lw=2, ls='--', label='$T_\\mathrm{eff}/T_\\odot$')
ax2.loglog(m_grid, ms_lifetime(m_grid),'steelblue',    lw=2, ls=':', label='$t_\\mathrm{MS}$ [Gyr]')
ax2.axvline(1.0, color='gold', lw=1.5, ls='--', label='$1\\,M_\\odot$')
ax2.set_xlabel('Stellar mass $m$ [$M_\\odot$]')
ax2.set_ylabel('Normalised quantity / t [Gyr]')
ax2.set_title('Main-sequence scaling relations')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('imf_ms_relations.png', dpi=150, bbox_inches='tight')
plt.show()

print('MS lifetimes — formula t = 10 × m^{-2.5} Gyr vs Maeder & Meynet (1989):')
print(f'{"Mass [M☉]":>10}  {"T_eff [K]":>10}  {"formula [Gyr]":>14}  {"M&M 1989 [Gyr]":>16}  {"type":>5}')
print('-' * 65)
mm89 = {0.8: 18, 1.0: 10, 1.5: 2.5, 2.0: 1.3, 3.0: 0.42,
        5.0: 0.098, 10: 0.022, 20: 0.0077, 40: 0.0048}
for m, spt in [(0.8,'K'),(1.0,'G'),(1.5,'F'),(2.0,'A'),(3.0,'B'),
               (5.0,'B'),(10,'B'),(20,'O'),(40,'O')]:
    t_form = ms_lifetime(m)
    t_lit  = mm89.get(m, '?')
    print(f'{m:>10.1f}  {ms_temperature(m):>10.0f}  {t_form:>14.3g}  {str(t_lit):>16}  {spt:>5}')
print()
print('Note: formula underestimates lifetimes for m > 20 M☉ (massive stars)')

In [ ]:
def ssp_spectrum(lam_nm, age_Gyr, m_min=0.1, m_max=120.0,
                  n_mass=300, imf='chabrier'):
    """
    Compute the integrated blackbody spectrum of a Simple Stellar Population.

    Parameters
    ----------
    lam_nm   : array  — wavelength grid [nm]
    age_Gyr  : float  — SSP age [Gyr]
    m_min    : float  — lower mass limit [M_sun]
    m_max    : float  — upper mass limit [M_sun]
    n_mass   : int    — number of logarithmic mass bins
    imf      : str    — 'chabrier' (default), 'kroupa', or 'salpeter'

    Returns
    -------
    spectrum : array  — integrated spectral flux density (arbitrary units)
    m_TO     : float  — main-sequence turnoff mass [M_sun] at this age
    """
    m_bins  = np.logspace(np.log10(m_min), np.log10(m_max), n_mass)
    dlogm   = np.diff(np.log10(m_bins), append=np.log10(m_bins[-1]) + np.log10(m_bins[-1]/m_bins[-2]))
    imf_fn  = {'chabrier': imf_chabrier,
               'kroupa':   imf_kroupa,
               'salpeter': imf_salpeter}[imf]

    lam_m   = lam_nm * 1e-9
    spectrum = np.zeros_like(lam_nm, dtype=float)
    m_TO    = m_min

    for m, dlm in zip(m_bins, dlogm):
        if ms_lifetime(m) < age_Gyr:
            continue                     # evolved off the MS
        m_TO = max(m_TO, m)

        T  = float(ms_temperature(m))
        R  = float(ms_radius(m))
        w  = imf_fn(m) * m * dlm        # weight ∝ dN/dlog(m) × m × dlm
        L  = R**2 * (T / T_sun)**4      # luminosity in L_sun

        # Blackbody normalised at 500 nm reference (avoid absolute flux issues)
        B    = B_lambda(lam_m, T)
        B_ref = float(B_lambda(500e-9, T_sun))
        spectrum += w * L * B / B_ref

    return spectrum, m_TO


# ── 8.4 Compute SSP spectra for four ages (Chabrier IMF, blackbody templates) ──
ages_Gyr   = [0.1, 1.0, 5.0, 10.0]
age_colors = ['mediumpurple', 'steelblue', 'goldenrod', 'firebrick']

lam_ssp_nm = np.linspace(100, 2500, 1500)
lam_ssp_m  = lam_ssp_nm * 1e-9

spectra  = {}
turnoffs = {}
for age in ages_Gyr:
    spec, m_to = ssp_spectrum(lam_ssp_nm, age)
    spectra[age]  = spec
    turnoffs[age] = m_to
    T_to = float(ms_temperature(m_to))
    print(f'Age = {age:4.1f} Gyr  |  m_TO = {m_to:.2f} M_sun  |  '
          f'T_TO = {T_to:.0f} K  |  λ_peak(TO) = {b_Wien/T_to*1e9:.0f} nm')

In [ ]:
# ── Plot integrated SSP spectra ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, (ax, norm) in enumerate(zip(axes, [True, False])):
    ax.axvspan(380, 700, color='lightyellow', alpha=0.6, zorder=0)

    for age, col in zip(ages_Gyr, age_colors):
        spec = spectra[age].copy()
        m_to = turnoffs[age]
        T_to = float(ms_temperature(m_to))

        if norm:
            peak_val = spec.max()
            if peak_val > 0:
                spec = spec / peak_val

        ax.plot(lam_ssp_nm, spec, color=col, lw=2,
                label=f't = {age:.1f} Gyr  ($m_\\mathrm{{TO}}={m_to:.1f}\\,M_\\odot$, '
                      f'$T_\\mathrm{{TO}}={T_to:.0f}$ K)')
        if norm:
            lam_pk = b_Wien / T_to * 1e9
            if 100 < lam_pk < 2500:
                ax.axvline(lam_pk, color=col, lw=0.8, ls='--', alpha=0.6)

    ax.set_xlabel('Wavelength $\\lambda$ [nm]')
    ax.set_xlim(100, 2500)
    ax.legend(fontsize=8.5)
    ax.xaxis.set_minor_locator(AutoMinorLocator())

    if norm:
        ax.set_ylabel('Normalised integrated flux')
        ax.set_title('SSP spectra (Kroupa IMF) — normalised\nDashed = turnoff-star Wien peak')
    else:
        ax.set_ylabel('Integrated flux (arb. units)')
        ax.set_title('SSP spectra (Kroupa IMF) — absolute\n'
                     'Young populations are much brighter (dominated by O/B stars)')
        ax.set_yscale('log')

plt.suptitle('Simple Stellar Population spectra: young (blue) vs old (red)', y=1.01)
plt.tight_layout()
plt.savefig('ssp_spectra.png', dpi=150, bbox_inches='tight')
plt.show()

print('Interpretation:')
print('  t = 0.1 Gyr: massive hot stars still alive → blue continuum, peaks in UV')
print('  t = 1.0 Gyr: B stars gone, A stars at turnoff → peak near 300-400 nm')
print('  t = 5.0 Gyr: turnoff near F0 (7000 K) → yellow-white spectrum')
print('  t = 10.0 Gyr: turnoff near G2 (solar) → red/orange spectrum')
print()
print('This is why old elliptical galaxies appear red and young starbursts are blue!')

In [ ]:
# ── 8.5  SSP spectrum using real MILES stellar spectra ──────────────────────
# Replaces Planck blackbodies with the observed spectra loaded in Section 7.
# Coverage: 350–743 nm (MILES wavelength range).
# `spectra_obs` must be in scope from the Section 7 cells.

# Build a sorted list of (Teff, sp_label, lam_nm, flux_norm) from spectra_obs
_tpls = sorted(
    [(teff, sp, lam_nm,
      flux / flux[(lam_nm >= 400) & (lam_nm <= 700)].max())
     for sp, (lam_nm, flux, sptype, teff) in spectra_obs.items()],
    key=lambda x: x[0]
)
_T_tpl = np.array([t[0] for t in _tpls])

print('MILES templates available (sorted by Teff):')
for t in _tpls:
    print(f'  sp={t[1]:3s}  Teff={t[0]:6d} K')


def _nearest_miles(T_eff):
    """Return (lam_nm, flux_norm) for the MILES template closest to T_eff."""
    idx = int(np.argmin(np.abs(_T_tpl - T_eff)))
    return _tpls[idx][2], _tpls[idx][3]


def ssp_miles(age_Gyr, m_min=0.5, m_max=30.0, n_mass=200, imf='chabrier'):
    """
    SSP integrated spectrum using MILES observed stellar spectra as templates.

    Parameters
    ----------
    age_Gyr : float  — SSP age [Gyr]
    m_min   : float  — lower mass cut [M_sun]
    m_max   : float  — upper mass cut [M_sun]
    imf     : str    — 'chabrier' (default), 'kroupa', or 'salpeter'

    Returns
    -------
    lam_nm   : array  — wavelength grid [nm]  (MILES range 350–743 nm)
    spectrum : array  — integrated flux (arb. units)
    m_TO     : float  — turnoff mass [M_sun]
    """
    m_bins = np.logspace(np.log10(m_min), np.log10(m_max), n_mass)
    dlogm  = np.diff(np.log10(m_bins),
                     append=np.log10(m_bins[-1]) + np.log10(m_bins[-1] / m_bins[-2]))
    imf_fn = {'chabrier': imf_chabrier,
              'kroupa':   imf_kroupa,
              'salpeter': imf_salpeter}[imf]

    lam_ref, _  = _nearest_miles(_T_tpl[0])
    spectrum    = np.zeros_like(lam_ref, dtype=float)
    m_TO        = m_min

    for m, dlm in zip(m_bins, dlogm):
        if ms_lifetime(m) < age_Gyr:
            continue
        m_TO = max(m_TO, m)

        T = float(ms_temperature(m))
        R = float(ms_radius(m))
        w = imf_fn(m) * m * dlm        # IMF weight
        L = R**2 * (T / T_sun)**4      # luminosity [L_sun]

        lam_tpl, f_norm = _nearest_miles(T)
        spectrum += w * L * f_norm

    return lam_ref, spectrum, m_TO


# ── Compute and plot ────────────────────────────────────────────────────────
ages_miles = [0.1, 1.0, 5.0, 10.0]
age_colors = ['mediumpurple', 'steelblue', 'goldenrod', 'firebrick']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax_idx, (ax, norm) in enumerate(zip(axes, [True, False])):
    ax.axvspan(380, 700, color='lightyellow', alpha=0.6, zorder=0)

    for age, col in zip(ages_miles, age_colors):
        lam_out, spec, m_to = ssp_miles(age)
        T_to = float(ms_temperature(m_to))

        if norm:
            peak = spec.max()
            if peak > 0:
                spec = spec / peak

        ax.plot(lam_out, spec, color=col, lw=2,
                label=f't = {age:.1f} Gyr  '
                      f'($m_\\mathrm{{TO}}={m_to:.1f}\\,M_\\odot$, '
                      f'$T_\\mathrm{{TO}}={T_to:.0f}$ K)')
        if norm:
            lam_pk = b_Wien / T_to * 1e9
            if 350 < lam_pk < 743:
                ax.axvline(lam_pk, color=col, lw=0.8, ls='--', alpha=0.6)

    ax.set_xlabel('Wavelength $\\lambda$ [nm]')
    ax.set_xlim(350, 743)
    ax.legend(fontsize=8.5)
    ax.xaxis.set_minor_locator(AutoMinorLocator())

    if norm:
        ax.set_ylabel('Normalised integrated flux')
        ax.set_title('§8.5 MILES SSP (Chabrier IMF) — normalised')
    else:
        ax.set_ylabel('Integrated flux (arb. units)')
        ax.set_title('§8.5 MILES SSP (Chabrier IMF) — absolute')
        ax.set_yscale('log')

fig.suptitle(
    'SSP spectra from MILES stellar templates (350–743 nm, Chabrier IMF)\n'
    'Real absorption features now present — compare with §8.4 Planck SSP',
    y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig('ssp_miles_spectra.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key differences vs §8.4 Planck SSP:')
print('  Young (t=0.1 Gyr, B/A turnoff): Balmer absorption series visible')
print('  Mid-age (t=1 Gyr, F turnoff):   Ca II H&K and G-band emerge')
print('  Old (t≥5 Gyr, G/K turnoff):     Ca II very strong; Na D appears')
print('  All ages: real atmospheric line blanketing reddens continuum vs Planck')

---

## 9. Observed Galaxy Spectra

The SSP models developed in §8 predict the spectral energy distributions of stellar populations. We now compare them against real **galaxy spectra** — stacked spectra of thousands of SDSS galaxies from the [`qmost_templates`](https://github.com/JohanComparat/qmost_templates) repository (Comparat et al. 2020).

Two canonical galaxy types stand out in large spectroscopic surveys:

- **Star-forming / Emission-Line Galaxy (ELG):** Blue continuum dominated by OB stars. Prominent nebular emission lines — **[OII] λ3727**, **Hβ λ4861**, **[OIII] λλ4959, 5007**, **Hα λ6563** — are excited by UV photons from young massive stars ionising surrounding gas.
- **Quiescent / Luminous Red Galaxy (LRG):** Old stellar population, no recent star formation. Red continuum with a strong **4000 Å break** and deep absorption features: **Ca H&K λλ3934, 3968**, **G-band λ4304**, **Mg b λ5175**, **Na D λ5893**.

### 9.1 SSP model fitting

Both the blackbody SSP (`ssp_spectrum`, §8.4) and the MILES SSP (`ssp_miles`, §8.5) are fit to each galaxy template by minimising χ² over a single free parameter — the SSP age — restricted to the MILES wavelength range 3600–7400 Å.

### 9.2 ppxf — Penalized Pixel-Fitting

[`ppxf`](https://pypi.org/project/ppxf/) (Cappellari & Emsellem 2004; Cappellari 2017) fits an observed spectrum as the convolution of stellar templates with a line-of-sight velocity distribution (LOSVD):

$$F_\lambda^\mathrm{obs}(\lambda) = \int \mathrm{LOSVD}(v)\, F_\lambda^\mathrm{tpl}\!\left(\lambda\,e^{-v/c}\right) dv + P(\lambda)$$

where $P(\lambda)$ is a low-order polynomial for the continuum. Fitting the LOSVD yields the stellar **radial velocity** $V$ and **velocity dispersion** $\sigma$. We apply ppxf to the LRG template (strongest absorption features) using the six MILES stellar templates from §7.

In [ ]:
# -- 9.1  Install ppxf and download galaxy spectral templates ----------------
import subprocess, sys

try:
    import ppxf as _ppxf_pkg
    print(f'ppxf {_ppxf_pkg.__version__} already installed.')
except ImportError:
    print('Installing ppxf...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ppxf', '-q'])
    import ppxf as _ppxf_pkg
    print(f'ppxf {_ppxf_pkg.__version__} installed.')

# Galaxy spectral stacks from qmost_templates (Comparat et al. 2020)
# https://github.com/JohanComparat/qmost_templates  --  z=0 rest-frame SDSS DR16 stacks
_GBASE = 'https://raw.githubusercontent.com/JohanComparat/qmost_templates/main/data/'
GALAXY_FILES = {
    'ELG': 'DR16_ELG-stitched-stack.fits',   # star-forming emission-line galaxy
    'LRG': 'DR16LRG-stitched-stack.fits',    # quiescent luminous red galaxy
}

galaxy_data = {}
for _gname, _gfname in GALAXY_FILES.items():
    if not os.path.exists(_gfname):
        print(f'Downloading {_gfname}...')
        urllib.request.urlretrieve(_GBASE + _gfname, _gfname)
        print(f'  Saved -> {_gfname}')
    with fits.open(_gfname) as hdul:
        try:
            d = hdul[1].data
        except IndexError:
            d = hdul[0].data
        wav  = d['wavelength'].astype(float)   # Angstrom, rest-frame z=0
        flux = d['medianStack'].astype(float)
    galaxy_data[_gname] = (wav, flux)
    print(f'{_gname}: {len(wav)} pixels   lambda = [{wav[0]:.0f} -- {wav[-1]:.0f}] Ang')

elg_wav, elg_flux = galaxy_data['ELG']
lrg_wav, lrg_flux = galaxy_data['LRG']

# -- Plot both templates with labeled spectral lines -------------------------
EMISSION = {'[OII] 3727': 3727, 'H-beta': 4861,
            '[OIII] 4959': 4959, '[OIII] 5007': 5007, 'H-alpha': 6563}
ABSORPT  = {'Ca K 3934': 3934, 'Ca H 3968': 3968,
            'G-band': 4304, 'Mg b': 5175, 'Na D': 5893}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=False)

for ax, (wav, flux), lines, lc, title, gal_col in [
    (ax1, (elg_wav, elg_flux), EMISSION, 'crimson',
     'Star-forming / Emission-Line Galaxy (ELG) -- rest-frame z=0', 'steelblue'),
    (ax2, (lrg_wav, lrg_flux), ABSORPT,  'navy',
     'Quiescent / Luminous Red Galaxy (LRG) -- rest-frame z=0',   'firebrick'),
]:
    m5060 = (wav > 5000) & (wav < 6000)
    norm  = np.nanmedian(flux[m5060]) if m5060.sum() > 0 else np.nanmedian(np.abs(flux))
    ax.plot(wav, flux / norm, color=gal_col, lw=1, alpha=0.9)
    ax.axvspan(3800, 7000, color='lightyellow', alpha=0.35, zorder=0)
    for i, (name, lam) in enumerate(lines.items()):
        ax.axvline(lam, color=lc, lw=1, ls='--', alpha=0.75)
        ax.text(lam + 28, 0.03 + 0.08 * (i % 2), name, fontsize=7.5,
                color=lc, rotation=90, va='bottom',
                transform=ax.get_xaxis_transform())
    ax.set_xlabel('Wavelength [Ang]')
    ax.set_ylabel('Flux (normalised 5000-6000 Ang)')
    ax.set_title(title)
    ax.xaxis.set_minor_locator(AutoMinorLocator())

# Mark 4000 Ang break on LRG panel
ax2.axvline(4000, color='darkgreen', lw=1.5, ls=':', alpha=0.85)
ax2.text(4020, 0.88, '4000 Ang break', color='darkgreen', fontsize=8.5,
         rotation=90, va='top', transform=ax2.get_xaxis_transform())

# Set wavelength ranges appropriate for each template
ax1.set_xlim(1750, 7000)    # ELG: UV continuum through H-alpha
ax2.set_xlim(2400, 11500)   # LRG: UV through near-IR

plt.tight_layout()
plt.savefig('galaxy_templates.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# -- 9.2  Fit blackbody and MILES SSP models to the galaxy templates ---------
from scipy.optimize import minimize_scalar

# Fixed wavelength grid for fitting: MILES coverage 3600-7400 Ang
LAM_FIT_A  = np.linspace(3600., 7400., 800)
LAM_FIT_NM = LAM_FIT_A / 10.0

def _normalise(flux, lam=None, lo=5000., hi=6000.):
    # Divide by median in 5000-6000 Ang window (or global median).
    if lam is not None:
        m = (lam >= lo) & (lam <= hi)
        val = np.nanmedian(flux[m]) if m.sum() > 0 else np.nanmedian(flux[flux > 0])
    else:
        val = np.nanmedian(flux[flux > 0])
    return flux / val if val and val != 0 else flux

def _chi2(obs, model):
    ok = np.isfinite(obs) & np.isfinite(model) & (model > 0)
    if ok.sum() < 30:
        return 1e10
    scale = np.median(obs[ok]) / np.median(model[ok])
    r = (obs[ok] - scale * model[ok]) / (np.median(np.abs(obs[ok])) + 1e-30)
    return float(np.mean(r**2))

# Interpolate galaxy templates onto fitting grid
elg_fit = np.interp(LAM_FIT_A, elg_wav, elg_flux, left=np.nan, right=np.nan)
lrg_fit = np.interp(LAM_FIT_A, lrg_wav, lrg_flux, left=np.nan, right=np.nan)

# Blackbody SSP chi2
def chi2_bb(log_age, flux_obs):
    ssp_fl, _ = ssp_spectrum(LAM_FIT_NM, 10**log_age)
    return _chi2(flux_obs, ssp_fl)

# MILES SSP chi2 -- get MILES grid once; restrict to fitting range
_lam_miles_nm0, _, _ = ssp_miles(1.0)
_lam_miles_A0 = _lam_miles_nm0 * 10.0
_mm = (_lam_miles_A0 >= 3600.) & (_lam_miles_A0 <= 7400.)
elg_on_miles = np.interp(_lam_miles_A0[_mm], elg_wav, elg_flux)
lrg_on_miles = np.interp(_lam_miles_A0[_mm], lrg_wav, lrg_flux)

def chi2_miles(log_age, flux_obs):
    _, ssp_fl, _ = ssp_miles(10**log_age)
    return _chi2(flux_obs, ssp_fl[_mm])

# Run minimisation
print('Fitting SSP ages (minimize_scalar, log10 age in [-1, 1.2])...\n')
fit_ages = {}
for gname, fb, fm in [('ELG', elg_fit, elg_on_miles),
                       ('LRG', lrg_fit, lrg_on_miles)]:
    r_bb = minimize_scalar(chi2_bb,    bounds=(-1, 1.2), method='bounded', args=(fb,))
    r_mi = minimize_scalar(chi2_miles, bounds=(-1, 1.2), method='bounded', args=(fm,))
    fit_ages[gname] = {'bb': 10**r_bb.x, 'miles': 10**r_mi.x}
    print(f'{gname}:  Blackbody SSP age = {10**r_bb.x:.2f} Gyr  '
          f'|  MILES SSP age = {10**r_mi.x:.2f} Gyr')

# -- Plot: observed data vs best-fit model -----------------------------------
fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharey=False)

for row, (gname, gal_col, wav, flux) in enumerate([
    ('ELG', 'steelblue', elg_wav, elg_flux),
    ('LRG', 'firebrick', lrg_wav, lrg_flux),
]):
    gal_n = _normalise(flux, lam=wav)
    age_bb    = fit_ages[gname]['bb']
    age_miles = fit_ages[gname]['miles']
    ssp_fl_bb, _   = ssp_spectrum(LAM_FIT_NM, age_bb)
    lam_mi, ssp_fl_mi, _ = ssp_miles(age_miles)
    lam_mi_A = lam_mi * 10.0

    for col, (model_lbl, model_col, lam_m, flux_m) in enumerate([
        ('Blackbody SSP', 'darkorange',  LAM_FIT_A,  _normalise(ssp_fl_bb,    lam=LAM_FIT_A)),
        ('MILES SSP',     'forestgreen', lam_mi_A,   _normalise(ssp_fl_mi,    lam=lam_mi_A)),
    ]):
        ax = axes[row, col]
        age = age_bb if col == 0 else age_miles
        ax.plot(wav,   gal_n,  color=gal_col,   lw=1,  alpha=0.8, label=f'{gname}')
        ax.plot(lam_m, flux_m, color=model_col, lw=2,  ls='--',
                label=f'{model_lbl}  t = {age:.2f} Gyr')
        ax.set_xlim(3400, 7600)
        ax.set_xlabel('Wavelength [Ang]')
        ax.set_ylabel('Normalised flux')
        ax.set_title(f'{gname} vs {model_lbl}  (best age = {age:.2f} Gyr)')
        ax.legend(fontsize=8)
        ax.xaxis.set_minor_locator(AutoMinorLocator())

plt.suptitle('Sec 9.1  Single-age SSP fits to observed galaxy templates', y=1.01)
plt.tight_layout()
plt.savefig('ssp_fit_galaxy.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInterpretation:')
print('  ELG best age < 2 Gyr  -> dominated by young hot stars (recent starburst)')
print('  LRG best age > 5 Gyr  -> old stellar population, passive evolution')
print('  MILES SSP captures real absorption features; BB SSP matches overall colour')

In [ ]:
# -- 9.3  ppxf stellar kinematics fit of the LRG template --------------------
# ppxf fits F_obs = LOSVD ⊗ F_template + polynomial continuum.
# We use the LRG (absorption-line dominated) with 6 MILES templates from Sec 7.

from ppxf.ppxf import ppxf
import ppxf.ppxf_util as util

# 1. Prepare LRG galaxy spectrum in the MILES wavelength range
miles_lam_A_ref = spectra_obs['G'][0] * 10.0          # nm -> Ang
miles_lam_range = np.array([miles_lam_A_ref[0], miles_lam_A_ref[-1]])

# Interpolate LRG onto a uniform 1.5 Ang grid spanning the MILES range
lam_unif_A = np.arange(miles_lam_range[0], miles_lam_range[1], 1.5)
lrg_unif   = np.interp(lam_unif_A, lrg_wav, lrg_flux)

galaxy, logLam_gal, velscale = util.log_rebin(miles_lam_range, lrg_unif)
noise = np.full_like(galaxy, np.abs(np.median(galaxy)) * 0.02 + 1e-10)

# 2. Interpolate 6 MILES templates onto the galaxy's log-wavelength grid
#    (equivalent to log-rebinning with the same velscale; avoids a numpy
#    compatibility issue in ppxf_util.log_rebin with velscale parameter)
lam_gal_grid = np.exp(logLam_gal)   # galaxy wavelengths in Ang
tpl_list = []
for sp in ['B', 'A', 'F', 'G', 'K', 'M']:
    lam_sp_nm, flux_sp, _, _ = spectra_obs[sp]
    lam_sp_A = lam_sp_nm * 10.0
    t = np.interp(lam_gal_grid, lam_sp_A, flux_sp, left=0., right=0.)
    med = np.median(t[t > 0]) if np.any(t > 0) else 1.
    tpl_list.append(t / med)

templates = np.column_stack(tpl_list)    # (n_pix_galaxy, 6)

# 3. Run ppxf
# moments=2 fits V and sigma (Gaussian LOSVD)
# degree=4:  4th-degree additive Legendre polynomial for continuum shape
start = [0., 150.]     # initial [V (km/s), sigma (km/s)]
pp = ppxf(templates, galaxy, noise, velscale, start,
          moments=2, degree=4, mdegree=0, quiet=True)

print('ppxf results  (LRG template, 6 MILES stars):')
print(f'  Stellar velocity   V = {pp.sol[0]:+.1f}  km/s')
print(f'  Velocity dispersion sigma = {pp.sol[1]:.1f}  km/s')
print(f'  Reduced chi^2        = {pp.chi2:.4f}')
print('  Template weights (B A F G K M):',
      '  '.join(f'{w:.3f}' for w in pp.weights))

# 4. Plot: spectrum + best-fit + residuals
lam_gal_A = np.exp(logLam_gal)
residuals  = galaxy - pp.bestfit

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(14, 8),
    gridspec_kw={'height_ratios': [3, 1]}, sharex=True
)

ax_top.plot(lam_gal_A, galaxy,     color='k',       lw=1,
            label='LRG spectrum (log-rebinned)')
ax_top.plot(lam_gal_A, pp.bestfit, color='crimson', lw=1.5,
            label=f'ppxf best fit  (V = {pp.sol[0]:+.0f} km/s, '
                  f'sigma = {pp.sol[1]:.0f} km/s, chi2 = {pp.chi2:.3f})')
ax_top.fill_between(lam_gal_A, galaxy, pp.bestfit, color='crimson', alpha=0.12)
ax_top.set_ylabel('Flux (arb. units)')
ax_top.set_title('Sec 9.2  ppxf fit: LRG spectrum with 6 MILES stellar templates')
ax_top.legend(fontsize=9)

ax_bot.axhline(0, color='k', lw=0.8, ls='--')
ax_bot.plot(lam_gal_A, residuals, color='steelblue', lw=0.8, label='Residuals')
ax_bot.set_xlabel('Wavelength [Ang]')
ax_bot.set_ylabel('Residuals')
ax_bot.legend(fontsize=8)

for name, lam in ABSORPT.items():
    if miles_lam_range[0] < lam < miles_lam_range[1]:
        for ax in (ax_top, ax_bot):
            ax.axvline(lam, color='navy', lw=0.7, ls=':', alpha=0.55)
        ax_top.text(lam + 20, 0.96, name, fontsize=7, color='navy',
                    rotation=90, va='top', transform=ax_top.get_xaxis_transform())

for ax in (ax_top, ax_bot):
    ax.xaxis.set_minor_locator(AutoMinorLocator())

plt.tight_layout()
plt.savefig('ppxf_lrg_fit.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nNote: sigma reflects combined instrumental broadening + intrinsic dispersion.')
print('For a real massive LRG, sigma ~ 200-350 km/s is typical.')

---

## 10. Exercises

### Exercise 1 — Wien's law: identifying a star from its flux ratio

A star's spectrum is measured at two wavelengths: the flux at 400 nm is three times larger than the flux at 800 nm, i.e. $F_{400}/F_{800} = 3$.

**(a)** Using the Planck function $B_\lambda$, solve numerically for the temperature $T$ that produces this ratio.

**(b)** What spectral type is this star? Where does its Wien peak lie?

**(c)** Plot the Planck function for your solution and mark the two measurement wavelengths.

---

### Exercise 2 — Stefan-Boltzmann: luminosity of a hot white dwarf

A white dwarf has $T_\mathrm{eff} = 50\,000$ K and radius $R = 0.012\,R_\odot$.

**(a)** Compute its luminosity using the Stefan–Boltzmann law.

**(b)** Where is its Wien peak? Is most of its emission observable from the ground?

**(c)** Despite its high temperature, why is this white dwarf much less luminous than an O5 star ($T = 42\,000$ K, $R = 12\,R_\odot$)? Quantify the difference.

---

### Exercise 3 — Spectral library: comparing blackbody and observed continuum

You are given the following flux measurements (in arbitrary units, continuum only, no lines) from a real spectrum at several wavelengths:

| $\lambda$ [nm] | Flux |
|:-:|:-:|
| 400 | 0.52 |
| 500 | 0.88 |
| 600 | 1.00 |
| 700 | 0.98 |
| 800 | 0.87 |
| 900 | 0.72 |

**(a)** Fit a Planck blackbody to these measurements by minimising the residuals (hint: use `scipy.optimize.minimize_scalar`). What is the best-fit $T_\mathrm{eff}$?

**(b)** What spectral type does this correspond to?

**(c)** In a real spectral library (MILES or MaStar), what additional features would you see on top of this continuum for a star of this type?

---

### Exercise 4 — Stellar population synthesis: colour evolution

Using the `ssp_spectrum` function defined in Section 8:

**(a)** Compute the SSP spectra for ages 0.01, 0.1, 1, 3, 10 Gyr.

**(b)** For each age, compute the synthetic "colour" as the logarithm of the ratio of flux at 440 nm to flux at 640 nm:
$$C = -2.5 \log_{10}\!\left(\frac{F_{440}}{F_{640}}\right)$$
This is analogous to a $B-R$ colour index.

**(c)** Plot $C$ versus age. What does the trend tell you about how the colour of a stellar population evolves over time?

In [ ]:
from scipy.optimize import brentq, minimize_scalar

# ── Exercise 1: find T from F(400)/F(800) = 3 ─────────────────────────────
target_ratio = 3.0
lam1_m, lam2_m = 400e-9, 800e-9

def flux_ratio(T_K):
    return B_lambda(lam1_m, T_K) / B_lambda(lam2_m, T_K) - target_ratio

T_ex1 = brentq(flux_ratio, 2000, 50000)
lam_peak_ex1 = b_Wien / T_ex1 * 1e9

print('Exercise 1: F(400 nm) / F(800 nm) = 3')
print(f'  Solution: T_eff = {T_ex1:.0f} K')
print(f'  Wien peak: lambda_max = {lam_peak_ex1:.0f} nm')
region = 'UV' if lam_peak_ex1 < 380 else ('Visible' if lam_peak_ex1 < 700 else 'Near-IR')
print(f'  Region: {region}')
for spt, (Teff, *_) in SPTYPES.items():
    if abs(Teff - T_ex1) < 1000:
        print(f'  Closest spectral type: {spt} (T_eff = {Teff} K)')

# Verification
ratio_check = B_lambda(lam1_m, T_ex1) / B_lambda(lam2_m, T_ex1)
print(f'  Check: B(400)/B(800) at T={T_ex1:.0f} K = {ratio_check:.3f}  (target = {target_ratio})')

# Plot
fig, ax = plt.subplots(figsize=(9, 4))
lam_ex1_nm = np.linspace(300, 1200, 1000)
B_ex1 = B_lambda(lam_ex1_nm * 1e-9, T_ex1)
ax.plot(lam_ex1_nm, B_ex1 / B_ex1.max(), 'steelblue', lw=2,
        label=f'$B_\\lambda(T={T_ex1:.0f}$ K)')
ax.axvspan(380, 700, color='lightyellow', alpha=0.6, zorder=0)
for lam_m_pt, lam_nm_pt, name in [(lam1_m, 400, '400 nm'), (lam2_m, 800, '800 nm')]:
    val = B_lambda(lam_m_pt, T_ex1) / B_ex1.max()
    ax.scatter([lam_nm_pt], [val], s=120, zorder=10, edgecolors='k', lw=1)
    ax.annotate(f'{name}\nflux={val:.2f}', xy=(lam_nm_pt, val),
                xytext=(lam_nm_pt + 40, val + 0.05), fontsize=9)
ax.axvline(lam_peak_ex1, color='steelblue', ls='--', lw=1,
           label=f'Wien peak = {lam_peak_ex1:.0f} nm')
ax.set_xlabel('Wavelength [nm]')
ax.set_ylabel('Normalised $B_\\lambda$')
ax.set_title(f'Exercise 1: Blackbody with $F(400)/F(800) = {target_ratio}$')
ax.legend()
plt.tight_layout()
plt.show()

print()
# ── Exercise 2: white dwarf luminosity ────────────────────────────────────
T_wd, R_wd = 50000.0, 0.012   # K, R_sun
L_wd_W, L_wd_ratio = stefan_boltzmann_luminosity(R_wd, T_wd)
lam_peak_wd = b_Wien / T_wd * 1e9

T_O5, R_O5 = 42000.0, 12.0
L_O5_W, L_O5_ratio = stefan_boltzmann_luminosity(R_O5, T_O5)

print('Exercise 2: Hot white dwarf')
print(f'  T_eff = {T_wd:.0f} K,  R = {R_wd} R_sun')
print(f'  L = {L_wd_W:.3e} W = {L_wd_ratio:.4f} L_sun')
print(f'  Wien peak: lambda_max = {lam_peak_wd:.1f} nm  (extreme UV — not visible from ground!)')
print()
print(f'  O5 star: T={T_O5:.0f} K, R={R_O5} R_sun → L = {L_O5_ratio:.0f} L_sun')
print(f'  Ratio L(O5)/L(WD) = {L_O5_ratio/L_wd_ratio:.1f}')
print(f'  Despite higher T, WD is {L_O5_ratio/L_wd_ratio:.0f}× less luminous than O5 due to R^2 factor')
print(f'  (R_WD/R_O5)^2 = ({R_wd}/{R_O5})^2 = {(R_wd/R_O5)**2:.6f}')

In [ ]:
# ── Exercise 3: fit a blackbody to measured continuum fluxes ───────────────
lam_obs_pts  = np.array([400, 500, 600, 700, 800, 900], dtype=float)  # nm
flux_obs_pts = np.array([0.52, 0.88, 1.00, 0.98, 0.87, 0.72], dtype=float)

def bb_residuals(log_T):
    T_try = 10**log_T
    B_try = B_lambda(lam_obs_pts * 1e-9, T_try)
    B_norm = B_try / B_try.max()
    f_norm = flux_obs_pts / flux_obs_pts.max()
    return np.sum((B_norm - f_norm)**2)

result = minimize_scalar(bb_residuals, bounds=(3.0, 4.8), method='bounded')
T_fit  = 10**result.x

print('Exercise 3: Blackbody fit to observed continuum fluxes')
print(f'  Best-fit T_eff = {T_fit:.0f} K')
print(f'  Wien peak: lambda_max = {b_Wien/T_fit*1e9:.0f} nm')

diffs = {spt: abs(Teff - T_fit) for spt, (Teff, *_) in SPTYPES.items()}
best_spt = min(diffs, key=diffs.get)
print(f'  Closest spectral type: {best_spt} (T_eff = {SPTYPES[best_spt][0]} K)')
print()
print(f'  For a {best_spt} star, real MILES/MaStar spectra would show:')
for line, note in [('Ca II H&K (393/397 nm)', 'moderate strength'),
                    ('G-band CH (431 nm)', 'moderate'),
                    ('H beta (486 nm)', 'weakening compared to A stars'),
                    ('Na I D (589 nm)', 'beginning to appear'),
                    ('H alpha (656 nm)', 'weak absorption')]:
    print(f'    - {line}: {note}')

fig, ax = plt.subplots(figsize=(9, 4))
lam_fit_nm = np.linspace(350, 1000, 500)
B_fit = B_lambda(lam_fit_nm * 1e-9, T_fit)
B_fit_norm = B_fit / B_fit.max()
f_norm_plot = flux_obs_pts / flux_obs_pts.max()

ax.axvspan(380, 700, color='lightyellow', alpha=0.6, zorder=0)
ax.plot(lam_fit_nm, B_fit_norm, 'steelblue', lw=2,
        label=f'Best-fit blackbody $T = {T_fit:.0f}$ K')
ax.scatter(lam_obs_pts, f_norm_plot, color='firebrick', s=80, zorder=10,
           edgecolors='k', lw=0.8, label='Observed continuum (normalised)')
ax.set_xlabel('Wavelength [nm]')
ax.set_ylabel('Normalised flux')
ax.set_title(f'Exercise 3: Blackbody fit → $T_\\mathrm{{eff}} = {T_fit:.0f}$ K ({best_spt})')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Exercise 4: SSP colour evolution ────────────────────────────────────────
ages_ex4 = [0.01, 0.05, 0.1, 0.3, 1.0, 3.0, 5.0, 10.0, 13.0]

lam_ex4_nm = np.linspace(100, 2500, 1500)
lam_ex4_m  = lam_ex4_nm * 1e-9

i_B = np.argmin(np.abs(lam_ex4_nm - 440))
i_R = np.argmin(np.abs(lam_ex4_nm - 640))

colours_ex4 = []
turnoffs_ex4 = []

print(f'{"Age [Gyr]":>10}  {"m_TO [Msun]":>12}  {"T_TO [K]":>10}  {"C (B-R analog)":>15}')
print('-' * 56)

for age in ages_ex4:
    spec, m_to = ssp_spectrum(lam_ex4_nm, age)
    F_B_val = spec[i_B]
    F_R_val = spec[i_R]
    if F_B_val > 0 and F_R_val > 0:
        C = -2.5 * np.log10(F_B_val / F_R_val)
    else:
        C = np.nan
    colours_ex4.append(C)
    turnoffs_ex4.append(m_to)
    T_to = float(ms_temperature(m_to))
    print(f'{age:>10.2f}  {m_to:>12.2f}  {T_to:>10.0f}  {C:>15.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.semilogx(ages_ex4, colours_ex4, 'ko-', lw=2, ms=7)
ax.set_xlabel('SSP Age [Gyr]')
ax.set_ylabel('$C = -2.5\\log(F_{440}/F_{640})$ [mag]')
ax.set_title('Exercise 4: SSP colour evolution (Kroupa IMF)\n'
             'Positive C = red; Negative C = blue')
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.fill_between(ages_ex4, 0, np.array(colours_ex4),
                where=np.array(colours_ex4) > 0, color='firebrick', alpha=0.15, label='Red')
ax.fill_between(ages_ex4, 0, np.array(colours_ex4),
                where=np.array(colours_ex4) <= 0, color='steelblue', alpha=0.15, label='Blue')
ax.legend(fontsize=9)

ax2 = axes[1]
ax2.scatter(turnoffs_ex4, colours_ex4, c=np.log10(ages_ex4),
            cmap='RdYlBu_r', s=80, zorder=10, edgecolors='k', lw=0.8)
for age, m_to, C in zip(ages_ex4, turnoffs_ex4, colours_ex4):
    ax2.annotate(f'{age:.2f} Gyr', xy=(m_to, C), xytext=(m_to + 0.3, C + 0.03),
                 fontsize=7, color='gray')
ax2.set_xlabel('Turnoff mass $m_\\mathrm{TO}$ [$M_\\odot$]')
ax2.set_ylabel('$C$ [mag]')
ax2.set_title('SSP colour vs turnoff mass')
ax2.axhline(0, color='gray', lw=0.8, ls='--')
ax2.invert_xaxis()

plt.tight_layout()
plt.savefig('ssp_colour_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

print()
print('Interpretation:')
print('  Young populations (age < 1 Gyr): hot O/B/A stars dominate → blue (negative C)')
print('  Old populations (age > 5 Gyr): turnoff near F/G → red (positive C)')
print('  This colour evolution matches what is observed in galaxies:')
print('    - Elliptical/lenticular galaxies (old stars): red')
print('    - Spiral/irregular galaxies (ongoing star formation): blue')